In [1]:
%%writefile exceptions.py

class UserManagementError(Exception):
    """Base exception for user management system errors."""
    pass

class UserExistsError(UserManagementError):
    """Exception raised when trying to register a user that already exists."""
    def __init__(self, username):
        self.username = username
        super().__init__(f"User '{username}' already exists.")

class UserNotFoundError(UserManagementError):
    """Exception raised when a user is not found."""
    def __init__(self, username):
        self.username = username
        super().__init__(f"User '{username}' not found.")

class InvalidCredentialsError(UserManagementError):
    """Exception raised for invalid username or password during login."""
    pass

class MaxLoginAttemptsExceededError(UserManagementError):
    """Exception raised when maximum login attempts are exceeded."""
    pass

class PermissionDeniedError(UserManagementError):
    """Exception raised when a user tries to access a restricted resource."""
    pass

class FileOperationError(UserManagementError):
    """Exception raised for general file operation errors."""
    pass


Writing exceptions.py


In [2]:
%%writefile log.py

import logging
import os
from datetime import datetime

LOG_DIR = 'logs'
LOG_FILE = os.path.join(LOG_DIR, f'user_management_{datetime.now().strftime("%Y%m%d")}.log')

# Ensure the log directory exists
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

def log_activity(level, message):
    """Logs an activity with the specified level and message."""
    if level == 'info':
        logging.info(message)
    elif level == 'warning':
        logging.warning(message)
    elif level == 'error':
        logging.error(message)
    elif level == 'critical':
        logging.critical(message)
    else:
        logging.debug(message)


Writing log.py


In [3]:
%%writefile utils.py

import hashlib
import base64

# Constants for password validation
MIN_PASSWORD_LENGTH = 8
HASHER = hashlib.sha256

def hash_password(password):
    """Hashes a password using SHA256."""
    return HASHER(password.encode('utf-8')).hexdigest()

def verify_password(stored_password_hash, provided_password):
    """Verifies a provided password against a stored hash."""
    return stored_password_hash == hash_password(provided_password)

def validate_password(password):
    """Validates password strength based on defined criteria.
    Returns True if valid, False otherwise."""
    if len(password) < MIN_PASSWORD_LENGTH:
        return False
    # Add more validation rules here (e.g., requires uppercase, lowercase, number, symbol)
    return True

def simple_encrypt(data, key='secretkey'):
    """A very simple XOR encryption for demonstration purposes."""
    encrypted_bytes = bytearray()
    key_bytes = key.encode('utf-8')
    data_bytes = data.encode('utf-8')
    for i in range(len(data_bytes)):
        encrypted_bytes.append(data_bytes[i] ^ key_bytes[i % len(key_bytes)])
    return base64.b64encode(encrypted_bytes).decode('utf-8')

def simple_decrypt(encrypted_data, key='secretkey'):
    """A very simple XOR decryption for demonstration purposes."""
    decrypted_bytes = bytearray()
    key_bytes = key.encode('utf-8')
    encrypted_bytes = base64.b64decode(encrypted_data.encode('utf-8'))
    for i in range(len(encrypted_bytes)):
        decrypted_bytes.append(encrypted_bytes[i] ^ key_bytes[i % len(key_bytes)])
    return decrypted_bytes.decode('utf-8')


Writing utils.py


In [4]:
%%writefile file_handler.py

import json
import os
import shutil
from datetime import datetime
from exceptions import FileOperationError
from log import log_activity

DATA_DIR = 'data'
USER_DATA_FILE = os.path.join(DATA_DIR, 'users.json')
BACKUP_DIR = os.path.join(DATA_DIR, 'backup')

# Ensure data and backup directories exist
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)
if not os.path.exists(BACKUP_DIR):
    os.makedirs(BACKUP_DIR)

def _load_data():
    """Loads user data from the JSON file."""
    if not os.path.exists(USER_DATA_FILE) or os.path.getsize(USER_DATA_FILE) == 0:
        return {}
    try:
        with open(USER_DATA_FILE, 'r') as f:
            return json.load(f)
    except json.JSONDecodeError as e:
        log_activity('error', f"Error decoding JSON from {USER_DATA_FILE}: {e}")
        raise FileOperationError(f"Corrupt user data file: {e}")
    except IOError as e:
        log_activity('error', f"Error reading {USER_DATA_FILE}: {e}")
        raise FileOperationError(f"Failed to read user data file: {e}")

def _save_data(data):
    """Saves user data to the JSON file."""
    try:
        with open(USER_DATA_FILE, 'w') as f:
            json.dump(data, f, indent=4)
    except IOError as e:
        log_activity('error', f"Error writing to {USER_DATA_FILE}: {e}")
        raise FileOperationError(f"Failed to write user data file: {e}")

def get_all_users():
    """Returns all user data."""
    return _load_data()

def save_all_users(users):
    """Saves the entire users dictionary to the file."""
    _save_data(users)

def backup_data():
    """Creates a timestamped backup of the user data file."""
    if os.path.exists(USER_DATA_FILE):
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        backup_file = os.path.join(BACKUP_DIR, f'users_backup_{timestamp}.json')
        try:
            shutil.copy(USER_DATA_FILE, backup_file)
            log_activity('info', f"User data backed up to {backup_file}")
            return True
        except IOError as e:
            log_activity('error', f"Failed to create backup: {e}")
            raise FileOperationError(f"Failed to create backup: {e}")
    log_activity('warning', "No user data file to backup.")
    return False


Writing file_handler.py


In [12]:
%%writefile auth.py

from file_handler import get_all_users, save_all_users, backup_data
from utils import hash_password, verify_password, validate_password
from log import log_activity
from datetime import datetime # Added this import
from exceptions import (
    UserExistsError,
    UserNotFoundError,
    InvalidCredentialsError,
    MaxLoginAttemptsExceededError,
    PermissionDeniedError,
    UserManagementError
)

class AuthService:
    MAX_LOGIN_ATTEMPTS = 3

    def __init__(self):
        self.current_user = None
        self.login_attempts = {}

    def register_user(self, username, password):
        users = get_all_users()
        if username in users:
            log_activity('warning', f"Registration attempt for existing user: {username}")
            raise UserExistsError(username)

        if not validate_password(password):
            log_activity('warning', f"Registration attempt with weak password for user: {username}")
            raise UserManagementError("Password does not meet strength requirements.")

        hashed_password = hash_password(password)
        users[username] = {
            'password': hashed_password,
            'registered_on': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        save_all_users(users)
        log_activity('info', f"User '{username}' registered successfully.")
        backup_data() # Backup after registration
        return True

    def login_user(self, username, password):
        users = get_all_users()
        if username not in users:
            log_activity('warning', f"Login attempt with non-existent user: {username}")
            raise UserNotFoundError(username)

        # Initialize login attempts for the user if not already present
        if username not in self.login_attempts:
            self.login_attempts[username] = 0

        if self.login_attempts[username] >= self.MAX_LOGIN_ATTEMPTS:
            log_activity('critical', f"Max login attempts exceeded for user: {username}")
            raise MaxLoginAttemptsExceededError(f"Maximum login attempts exceeded for user '{username}'. Account locked.")

        stored_password_hash = users[username]['password']
        if verify_password(stored_password_hash, password):
            self.current_user = username
            self.login_attempts[username] = 0 # Reset attempts on successful login
            log_activity('info', f"User '{username}' logged in successfully.")
            return True
        else:
            self.login_attempts[username] += 1
            remaining_attempts = self.MAX_LOGIN_ATTEMPTS - self.login_attempts[username]
            log_activity('warning', f"Invalid password attempt for user: {username}. Remaining attempts: {remaining_attempts}")
            raise InvalidCredentialsError(f"Invalid password. {remaining_attempts} attempts remaining.")

    def logout_user(self):
        if self.current_user:
            log_activity('info', f"User '{self.current_user}' logged out.")
            self.current_user = None
            return True
        return False

    def get_current_user_info(self):
        if not self.current_user:
            raise PermissionDeniedError("No user is currently logged in.")
        users = get_all_users()
        user_info = users.get(self.current_user, {})
        # Don't return the password hash
        display_info = {k: v for k, v in user_info.items() if k != 'password'}
        log_activity('info', f"Retrieved info for user: {self.current_user}")
        return display_info

    def update_user(self, username, new_password=None):
        if not self.current_user or self.current_user != username:
            raise PermissionDeniedError("You can only update your own account.")

        users = get_all_users()
        if username not in users:
            raise UserNotFoundError(username)

        if new_password:
            if not validate_password(new_password):
                raise UserManagementError("New password does not meet strength requirements.")
            users[username]['password'] = hash_password(new_password)
            users[username]['last_updated'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            save_all_users(users)
            log_activity('info', f"User '{username}' password updated successfully.")
            backup_data()
            return True
        return False

    def delete_user(self, username):
        if not self.current_user or self.current_user != username:
            raise PermissionDeniedError("You can only delete your own account.")

        users = get_all_users()
        if username not in users:
            raise UserNotFoundError(username)

        del users[username]
        save_all_users(users)
        self.logout_user() # Log out the deleted user
        log_activity('info', f"User '{username}' deleted successfully.")
        backup_data()
        return True

Overwriting auth.py


In [10]:
%%writefile main.py

from auth import AuthService
from exceptions import (
    UserExistsError,
    UserNotFoundError,
    InvalidCredentialsError,
    MaxLoginAttemptsExceededError,
    PermissionDeniedError,
    UserManagementError
)
from log import log_activity

def display_menu():
    print("\n--- User Management System ---")
    print("1. Register")
    print("2. Login")
    if auth_service.current_user:
        print("3. Dashboard")
        print("4. Update Account")
        print("5. Delete Account")
        print("6. Logout")
    print("0. Exit")
    print("------------------------------")

def register_user_cli():
    username = input("Enter username: ")
    password = input("Enter password: ")
    try:
        auth_service.register_user(username, password)
        print("Registration successful!")
    except UserManagementError as e:
        print(f"Error: {e}")

def login_user_cli():
    username = input("Enter username: ")
    password = input("Enter password: ")
    try:
        if auth_service.login_user(username, password):
            print(f"Welcome, {username}!")
    except (UserNotFoundError, InvalidCredentialsError, MaxLoginAttemptsExceededError) as e:
        print(f"Login failed: {e}")

def dashboard_cli():
    if not auth_service.current_user:
        print("Please log in first.")
        return

    try:
        user_info = auth_service.get_current_user_info()
        print("\n--- User Dashboard ---")
        for key, value in user_info.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
        print("----------------------")
    except PermissionDeniedError as e:
        print(f"Error: {e}")

def update_account_cli():
    if not auth_service.current_user:
        print("Please log in first.")
        return

    print(f"Updating account for {auth_service.current_user}.")
    new_password = input("Enter new password (leave blank to keep current): ")
    if new_password:
        try:
            auth_service.update_user(auth_service.current_user, new_password)
            print("Password updated successfully!")
        except UserManagementError as e:
            print(f"Error updating password: {e}")
    else:
        print("No changes made.")

def delete_account_cli():
    if not auth_service.current_user:
        print("Please log in first.")
        return

    confirm = input(f"Are you sure you want to delete account '{auth_service.current_user}'? (yes/no): ").lower()
    if confirm == 'yes':
        try:
            auth_service.delete_user(auth_service.current_user)
            print("Account deleted successfully. You have been logged out.")
        except UserManagementError as e:
            print(f"Error deleting account: {e}")
    else:
        print("Account deletion cancelled.")

def logout_user_cli():
    if auth_service.logout_user():
        print("Logged out successfully.")
    else:
        print("No user was logged in.")

# Initialize the authentication service
auth_service = AuthService()

def main():
    while True:
        display_menu()
        choice = input("Enter your choice: ")

        if choice == '1':
            register_user_cli()
        elif choice == '2':
            login_user_cli()
        elif choice == '3':
            if auth_service.current_user: # Only show dashboard if logged in
                dashboard_cli()
            else:
                print("Invalid choice. Please log in.")
        elif choice == '4':
            if auth_service.current_user:
                update_account_cli()
            else:
                print("Invalid choice. Please log in.")
        elif choice == '5':
            if auth_service.current_user:
                delete_account_cli()
            else:
                print("Invalid choice. Please log in.")
        elif choice == '6':
            if auth_service.current_user:
                logout_user_cli()
            else:
                print("Invalid choice. Please log in.")
        elif choice == '0':
            print("Exiting User Management System. Goodbye!")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()


Overwriting main.py


In [11]:
!python main.py


--- User Management System ---
1. Register
2. Login
0. Exit
------------------------------
Enter your choice: 1
Enter username: jaa
Enter password: jaa@1234
Traceback (most recent call last):
  File "/content/main.py", line 133, in <module>
    main()
  File "/content/main.py", line 103, in main
    register_user_cli()
  File "/content/main.py", line 29, in register_user_cli
    auth_service.register_user(username, password)
  File "/content/auth.py", line 34, in register_user
    'registered_on': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                     ^^^^^^^^
NameError: name 'datetime' is not defined. Did you forget to import 'datetime'?


In [13]:
!python main.py


--- User Management System ---
1. Register
2. Login
0. Exit
------------------------------
Enter your choice: 1
Enter username: jaa
Enter password: jaa@1234
2026-05-10 16:04:22,298 - INFO - User 'jaa' registered successfully.
2026-05-10 16:04:22,299 - INFO - User data backed up to data/backup/users_backup_20260510160422.json
Registration successful!

--- User Management System ---
1. Register
2. Login
0. Exit
------------------------------
Enter your choice: 2
Enter username: jaa
Enter password: jaa@1234
2026-05-10 16:04:46,288 - INFO - User 'jaa' logged in successfully.
Welcome, jaa!

--- User Management System ---
1. Register
2. Login
3. Dashboard
4. Update Account
5. Delete Account
6. Logout
0. Exit
------------------------------
Enter your choice: 3
2026-05-10 16:05:05,055 - INFO - Retrieved info for user: jaa

--- User Dashboard ---
Registered On: 2026-05-10 16:04:22
----------------------

--- User Management System ---
1. Register
2. Login
3. Dashboard
4. Update Account
5. Dele